In [1]:
import sys
import time

In [2]:
sys.path.append("../..")

In [3]:
from matplotlib import pyplot as plt 
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
from torchvision import transforms
from torch.nn import functional as F

from xaikd import models, datasets, utils, attributors
from xaikd.utils import metrics

from zennit.torchvision import ResNetCanonizer
from zennit.composites import EpsilonGammaBox
from zennit.attribution import Gradient

from torch.utils.data import random_split

from scipy.stats import ortho_group
from datetime import datetime

In [4]:
device = utils.get_device()
device

'cuda'

In [5]:
dataset = datasets.construct("imagenet-butterfly")

In [6]:
ds_train = dataset.create_subset(train_split=True)
ds_val = dataset.create_subset(train_split=False)

We have 7800 images in classes [321, 322, 323, 324, 325, 326]


preparing `ImageNetButterfly` samples: 100%|██████████| 7800/7800 [00:00<00:00, 2355841.52it/s]


We have 300 images in classes [321, 322, 323, 324, 325, 326]


preparing `ImageNetButterfly` samples: 100%|█████████████| 300/300 [00:00<00:00, 418314.89it/s]


In [7]:
MODEL_NAME = "imagenet-vgg16-tv"

MAIN_LAYER = "features.23"

In [8]:
teacher = models.get_trained_model(MODEL_NAME)
utils.modify_last_layer_for_subclasses(teacher, dataset.selected_classes)

teacher.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [9]:
teacher

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [10]:

# def last_layer_rewriting(model_name, rewriting):
    
#     arr_logits = []
#     arr_feats = []
#     arr_targets = []
    
#     model = models.get_trained_model(model_name)
#     utils.modify_last_layer_for_subclasses(model, dataset.selected_classes)
#     model.to(device)
    
#     if not rewriting:
#         print(f"[rewriting={rewriting}] Return Original Model")
#         return model
    
#     with torch.no_grad():
        
#         for count, (x, y) in tqdm(enumerate(datasets.build_dataloader(ds_train, shuffle=False))):
#             x = x.to(device)

#             logits = model(x).cpu().numpy()
#             arr_logits.append(logits)
#             arr_targets.append(np.argmax(logits, axis=1))

# #             if count > 1:
# #                 break
    
#     weights = model.__last_layer.weight.detach().cpu().numpy()
    
#     acc_before, _ = metrics.accuracy(
#         model,
#         dataloader=datasets.build_dataloader(ds_val, shuffle=False),
#         num_classes=dataset.num_classes,
#         device=device,
#     )

#     normed_weights = weights / np.linalg.norm(weights, axis=1, keepdims=True)
    
#     print(f"model_name={model_name}")
#     if "vgg" in model_name:
#         model.classifier[6] = torch.nn.Identity()
#     elif "resnet" in model_name:
#         model.fc = torch.nn.Identity()
#     else:
#         raise
    

#     with torch.no_grad():
        
#         for count, (x, y) in tqdm(enumerate(datasets.build_dataloader(ds_train, shuffle=False))):
#             x = x.to(device)

#             feat = model(x).cpu().numpy()
#             arr_feats.append(feat)
# #             if count > 1:
# #                 break

#     arr_logits = np.vstack(arr_logits)

#     arr_feats = np.vstack(arr_feats)
#     arr_targets = np.concatenate(arr_targets)
    
    
#     arr_w = []
#     arr_b = []
        
#     for class_ix in tqdm(sorted(np.unique(arr_targets))):
#         cond = arr_targets == class_ix
#         arr_class_feats = arr_feats[cond, :]
#         _, eigvecs = np.linalg.eigh(arr_class_feats.T @ arr_class_feats)
#         pc1 = eigvecs[:, -1]
#         pc1 = pc1 / np.linalg.norm(pc1)
        
#         proj = np.mean(arr_class_feats @ pc1) 
#         if proj < 0:
#             pc1 = -pc1
            
#         arr_w.append(pc1)
        
#         bias = np.mean(arr_feats @ pc1 - arr_logits[:, class_ix])
#         arr_b.append(bias)
        
    
#     out_feats, in_feats = normed_weights.shape
#     new_layer = torch.nn.Linear(out_features=out_feats, in_features=in_feats)
#     new_layer.weight = torch.nn.Parameter(
#         torch.from_numpy(np.stack(arr_w))
#     )
    
#     new_layer.bias = torch.nn.Parameter(
#         torch.tensor(arr_b)
#     )
    
#     if "vgg" in model_name:
#         model.classifier[6] = new_layer
#     elif "resnet" in model_name:
#         model.fc = new_layer
#     else:
#         raise

#     new_layer.to(device)
    
#     acc_after, _ = metrics.accuracy(
#         model,
#         dataloader=datasets.build_dataloader(ds_val, shuffle=False),
#         num_classes=dataset.num_classes,
#         device=device,
#     )
    
#     print(f"acc_before={acc_before}  : acc_after={acc_after}")
    
#     return model
            


# # teacher = last_layer_rewriting(MODEL_NAME, rewriting=True);

In [11]:
# teacher

In [12]:
teacher

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [13]:
teacher_acc, _ = metrics.accuracy(
    teacher,
    dataloader=datasets.build_dataloader(ds_val, shuffle=False),
    num_classes=dataset.num_classes,
    device=device,
)
teacher_acc

0.9599999785423279

In [14]:
def make_attributor_for(
    model,
    input_statistics,
    gamma,
    epsilon,
):
    input_transform = transforms.Normalize(*input_statistics)

    low, high = input_transform(torch.tensor([[[[[0.0]]] * 3], [[[[1.0]]] * 3]]))

    if isinstance(model, models.resnet.resnet.ResNet):
        canonizers = [ResNetCanonizer()]
    else:
        canonizers = []

    print(f"Instantiating EpsilonGammaBox(gamma={gamma}, eps={epsilon})")

    composite = EpsilonGammaBox(low=low, high=high, canonizers=canonizers, gamma=gamma, epsilon=epsilon)

    attributor = Gradient(
        model=model, 
        composite=composite
    )

    return attributor


    
def extract_activation_context(
    model,
    layer,
    dataset,
    data_loader,
    logit_modifier,
    rng,
    gamma=0.25,
    eps=1e-6,
    device="cpu",
    number_of_selected_spatial_locations=20,
    verbose=False
):
    arr_act = []
    arr_ctx = []

    
    with make_attributor_for(model, dataset.input_statistics, gamma=gamma, epsilon=eps) as attributor:
        for batch in tqdm(data_loader):
            x, y = batch
            x = x.to(device)

            try:
                module, hook = utils.interceptor.attach_hook_intercept_layer_output(
                    model, layer, should_retain_grad=True
                )

                _ = attributor.forward(x, lambda logits: logit_modifier(logits, y))

                act = utils.interceptor.get_output(module)
                rel = act.grad
                

                output_dimensions = act.shape[1:]
                # print("output.dimension", output_dimensions)
                # todo: check this with Gregoire again!
                ctx = torch.where(
                    act.abs() > 0, 
                    rel / act, 
                    0
                )

                np.testing.assert_allclose(
                    (act * ctx).detach().cpu().numpy(), 
                    (rel * (act.abs() > 0)).detach().cpu().numpy(),
                    atol=1e-6
                )

                assert ctx.shape == act.shape

                act = act.detach().cpu().numpy()
                ctx = ctx.detach().cpu().numpy()

                if len(act.shape) == 2:
                    act = act[:, :, None, None]
                    ctx = ctx[:, :, None, None]

                selected_act, selected_ctx = utils.subsample_tensors(
                    act,
                    ctx,
                    num_locations=number_of_selected_spatial_locations,
                    rng=rng,
                )
                arr_act.append(selected_act)
                arr_ctx.append(selected_ctx)

            finally:
                hook.remove()
    if verbose:
        print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)

    return arr_act, arr_ctx


GAMMA = 0.25

def get_act_ctx(model, dataset, layer, gamma, eps=1e-6, verbose=False):

    logit_modifier = attributors.WinningClassOneHotEvidence(num_classes=len(dataset.selected_classes))

    rng = np.random.default_rng(seed=1)

    dl = datasets.build_dataloader(
        ds_train,
        shuffle=False
    )
    return extract_activation_context(
        model=model,
        layer=layer,
        dataset=dataset,
        data_loader=dl,
        logit_modifier=logit_modifier,
        rng=rng,
        device=device,
        gamma=gamma,
        eps=eps,
        verbose=verbose,
    )


_arr_act, _arr_ctx = get_act_ctx(
    teacher,
    dataset,
    gamma=GAMMA,
    layer=MAIN_LAYER, 
    verbose=False
)

Instantiating EpsilonGammaBox(gamma=0.25, eps=1e-06)


100%|████████████████████████████████████████████████████████| 122/122 [01:33<00:00,  1.30it/s]


In [15]:
class BasisTransform:
    def rank_k_encoder(self, k: int):
        raise
    
    def rank_k_decoder(self, k: int):
        raise
        
    def get_hook_rank_k_transformation(self, k, device):
        
        mat_enc = self.rank_k_encoder(k)
        mat_dec = self.rank_k_decoder(k)

        # X @ U @ U.T
        mat = torch.from_numpy(mat_enc @ mat_dec)

        mat = mat.unsqueeze(2).unsqueeze(3).to(device)
        mean = torch.from_numpy(self.mean).reshape(1, -1, 1, 1).to(device)
        
        def hook_func(module, inp, out):
            return F.conv2d(
                out - mean, 
                mat
            ) + mean

        return hook_func
    
class PCA(BasisTransform):
    def __init__(self, arr_act, arr_ctx):
        self.mean = arr_act.mean(axis=0)
        eigvals, eigvecs = np.linalg.eigh(
            (arr_act - self.mean).T @ (arr_act - self.mean)
        )
        # descending sort
        sorted_indices = np.argsort(-eigvals)
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    def rank_k_encoder(self, k: int):
        # X @ U
        return self.eigvecs[:, :k]
    
    def rank_k_decoder(self, k: int):
        # Z @ U.T
        return self.eigvecs[:, :k].T


    def __str__(self):
        return "PCA"

PCA(_arr_act, _arr_ctx)

In [17]:
class PRCASortAbs(BasisTransform):
    def __init__(self, arr_act, arr_ctx):
        self.mean = arr_act.mean(axis=0)
        
        arr_centered_act = (arr_act - self.mean)
        ccov = arr_centered_act.T @ arr_ctx  + arr_ctx.T @ arr_centered_act
        
        eigvals, eigvecs = np.linalg.eigh(ccov)
        
        # descending sort
        sorted_indices = np.argsort(-np.abs(eigvals))
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    def rank_k_encoder(self, k: int):
        # X @ U
        return self.eigvecs[:, :k]
    
    def rank_k_decoder(self, k: int):
        # Z @ U.T
        return self.eigvecs[:, :k].T


    def __str__(self):
        return "PRCASortAbs"
PRCASortAbs(_arr_act, _arr_ctx)

(156000, 512)


In [25]:
class PRCA(BasisTransform):
    def __init__(self, arr_act, arr_ctx):
        self.mean = arr_act.mean(axis=0)
        
        arr_centered_act = (arr_act - self.mean)
        ccov = arr_centered_act.T @ arr_ctx  + arr_ctx.T @ arr_centered_act
        
        eigvals, eigvecs = np.linalg.eigh(ccov)
        
        # descending sort
        sorted_indices = np.argsort(-eigvals)
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    def rank_k_encoder(self, k: int):
        # X @ U
        return self.eigvecs[:, :k]
    
    def rank_k_decoder(self, k: int):
        # Z @ U.T
        return self.eigvecs[:, :k].T


    def __str__(self):
        return "PRCA"

In [18]:
class ProbPCA(BasisTransform):
    def __init__(self, arr_act, arr_ctx):
        self.mean = arr_act.mean(axis=0)
        eigvals, eigvecs = np.linalg.eigh(
            (arr_act - self.mean).T @ (arr_act - self.mean)
        )
        # descending sort
        sorted_indices = np.argsort(-eigvals)
        self.eigvals = eigvals[sorted_indices]
        self.eigvecs = eigvecs[:, sorted_indices]
    
    
    def rank_k_encoder(self, k: int):
        # todo
        pass

    def rank_k_decoder(self, k: int):
        # todo
        pass
    def __str__(self):
        return "ProbPCA"

In [19]:
def compute_accuracy_with_basis(
    model, layer, basis, k, dataset, dataloader
):
    module = utils.interceptor.get_module(model, layer)

    hook_func = basis.get_hook_rank_k_transformation(k, device=device)
    hook = module.register_forward_hook(hook_func)

    try:
        acc, _ = metrics.accuracy(
            model,
            dataloader=dataloader,
            num_classes=dataset.num_classes,
            device=device,
        )
        
    finally:
        hook.remove()

    return acc  

In [ ]:
import matplotlib as mpl


def compute_accuracy_of_basis_at_k(
    model, dataset, layer, 
    arr_ks,
    arr_bases,
):
    dl_val = datasets.build_dataloader(ds_val, shuffle=False)
    
    rows = []
    print(arr_bases)
    for k in tqdm(arr_ks):
        for basis in arr_bases:
    
            acc_val = compute_accuracy_with_basis(
                model=model, 
                layer=layer, 
                basis=basis,
                k=k,
                dataset=dataset, 
                dataloader=dl_val
            )

            rows.append(
                dict(
                    layer=layer, k=k, 
                    basis_name=f"{basis}",
                    acc_val=acc_val,
                )
            )
    

    df = pd.DataFrame(rows)

    return df


def estimate_rank_k_accuracies(
    model, dataset, layer,
    arr_bases,
    arr_ks=[1,2, 4, 8, 16, 32, 48, 64],
):

    colors = mpl.cm.Blues
    nbases = len(arr_bases)

    df = compute_accuracy_of_basis_at_k(
        model, dataset, layer, 
        arr_bases=arr_bases,
        arr_ks=arr_ks
    )

    torch.cuda.empty_cache()

    def alias(name):
        return name
        if name == "pca":
            return "PCA"
        elif "prcaopt:" in name:
            _, slug = name.split(":")
            return f"PCA-LH:{slug}"
        else:
            raise
            
    def ls(name):
        if name in  ["pca", "prca"]:
            return "--"
        else:
            return "-"

    def color(name):
        if name == "pca":
            return "blue"
        elif name == "prca":
            return "red"
        else:
            return None
            

    cols = [
        "acc_val"
    ]
    ncols = len(cols)
    
    plt.figure(figsize=(4*ncols, 3))
    plt.suptitle(f"{MODEL_NAME} layer={layer} (gamma={GAMMA})\n({dataset.selected_classes})", y=1.1)

    for cix, col in enumerate(cols):
        plt.subplot(1, ncols, cix+1)
        plt.title(col)
        plt.axhline(teacher_acc, ls="--", color="black", label=f"teacher_acc={teacher_acc:.4f}")
        for nix, name in enumerate(df.basis_name.unique()):
            _df = df[df.basis_name == name]
            
            plt.plot(
                _df.k, 
                _df[col],
                label=alias(name),
                ls=ls(name),
                marker=".",
            )
    
        plt.ylim([0, 1])
        plt.legend()
        plt.ylabel("Accuracy"); plt.xlabel("k")
        plt.xticks(df.k.unique())
    
    return df
    
estimate_rank_k_accuracies(
    model=teacher,
    dataset=dataset,
    arr_bases=[
        PCA(_arr_act, _arr_ctx),
        PRCASortAbs(_arr_act, _arr_ctx),
        PRCA(_arr_act, _arr_ctx)
    ],
    layer=MAIN_LAYER
)

(156000, 512)


In [ ]:
print(f"Finished at {datetime.now()}")